# Pemrosesan Imagery: Hybrid Approach (Pixel & Object-Based) & Real Pyramids (.ovr)

Notebook ini mengimplementasikan dua fitur utama:
1. **Hybrid Approach Background Removal**:
   - **Jika ada Alpha Channel**: Menggunakan **Pixel-based** (Otsu Thresholding) yang cepat dan presisi tinggi.
   - **Jika hanya 3-Band (RGB murni)**: Menggunakan **Object-based (Border-Connected Components)** untuk menghapus background putih di tepi kanvas tanpa menghapus awan putih di dalam daratan.
2. **Pembangunan Piramida Sejati (`.ovr`) untuk ArcGIS & QGIS**:
   - Membangun piramida eksternal standar ArcGIS/QGIS (`.ovr`) dengan level lengkap powers-of-2 `[2, 4, 8, 16, 32, 64, 128, 256, 512]`.
   - ArcGIS dan QGIS akan langsung mengenali piramida ini tanpa pernah memunculkan dialog/peringatan "Build Pyramids" lagi.

In [ ]:
import os
import glob
from pathlib import Path
import rasterio
from rasterio.windows import Window
from rasterio.enums import ColorInterp, Resampling
import numpy as np
import scipy.ndimage as ndi
import matplotlib.pyplot as plt
from tqdm import tqdm

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

print("Environment siap!")
print(f"Rasterio : {rasterio.__version__}")
print(f"NumPy    : {np.__version__}")

## 1. Definisi Model Hybrid Approach (Pixel-Based + Object-Based)
Algoritma secara otomatis memilih metode terbaik berdasarkan karakteristik citra:
- **Pixel-based (Alpha Otsu)** jika citra memiliki Band 4 (Alpha).
- **Object-based (Border-Connected)** jika citra berupa RGB murni agar awan putih di daratan tidak ikut terpotong.

In [ ]:
def otsu_threshold(arr):
    """Menghitung ambang batas pemisah optimal menggunakan Otsu algorithm."""
    hist, _ = np.histogram(arr, bins=256, range=(0, 256))
    total = arr.size
    sum_total = np.dot(np.arange(256), hist)
    sum_b, w_b, current_max, threshold = 0, 0, 0, 128
    for i in range(256):
        w_b += hist[i]
        if w_b == 0:
            continue
        w_f = total - w_b
        if w_f == 0:
            break
        sum_b += i * hist[i]
        m_b = sum_b / w_b
        m_f = (sum_total - sum_b) / w_f
        v = w_b * w_f * ((m_b - m_f) ** 2)
        if v > current_max:
            current_max, threshold = v, i
    return threshold

def hybrid_detect_background(filepath, factor=64):
    """
    Model Hybrid:
    - Mendeteksi ketersediaan kanal Alpha.
    - Menggunakan Pixel-Based (Alpha Otsu) jika ada Alpha.
    - Menggunakan Object-Based (Border-Connected Components) jika RGB murni.
    """
    with rasterio.open(filepath) as src:
        shape = (src.count, max(1, src.height // factor), max(1, src.width // factor))
        thumb = src.read(out_shape=shape)
        
        # Analisis piksel tepi kanvas luar (perimeter 5 piksel)
        top = thumb[:, :5, :]
        bot = thumb[:, -5:, :]
        left = thumb[:, :, :5]
        right = thumb[:, :, -5:]
        borders = np.concatenate([
            top.reshape(src.count, -1),
            bot.reshape(src.count, -1),
            left.reshape(src.count, -1),
            right.reshape(src.count, -1)
        ], axis=1)
        border_vals = np.round(np.median(borders, axis=1)).astype(int)
        
        has_alpha = False
        alpha_thresh = None
        
        # Evaluasi Band 4 sebagai Alpha Channel
        if src.count >= 4:
            if border_vals[3] <= 25 and np.percentile(thumb[3], 85) > 150:
                has_alpha = True
                alpha_thresh = otsu_threshold(thumb[3])
        
        profile = {
            'file': filepath,
            'bands': src.count,
            'dtype': src.dtypes[0],
            'width': src.width,
            'height': src.height,
            'crs': str(src.crs),
            'border_values': border_vals.tolist(),
            'has_alpha_channel': has_alpha,
            'alpha_threshold': alpha_thresh
        }
        
        if has_alpha:
            # STRATEGI 1: PIXEL-BASED PADA ALPHA (OTSU)
            profile['approach'] = 'Pixel-Based (Alpha Channel with Otsu)'
            profile['mask_rule'] = f'Band 4 (Alpha) >= {alpha_thresh}'
            profile['get_valid_mask'] = lambda data: (data[3] >= alpha_thresh)
        else:
            # STRATEGI 2: OBJECT-BASED PADA RGB (BORDER CONNECTIVITY)
            # Mencegah awan putih di daratan terhapus
            profile['approach'] = 'Object-Based (Border-Connected Components)'
            is_white = bool(np.all(border_vals[:3] >= 240))
            if is_white:
                profile['mask_rule'] = 'Object-Based: Outer White Border Only (Awan Daratan Terlindungi)'
                def get_object_mask_white(data):
                    white_cand = (data[0] >= 245) & (data[1] >= 245) & (data[2] >= 245)
                    labeled, _ = ndi.label(white_cand)
                    border_labels = set(labeled[0, :]).union(set(labeled[-1, :])).union(set(labeled[:, 0])).union(set(labeled[:, -1]))
                    border_labels.discard(0)
                    is_outer_bg = np.isin(labeled, list(border_labels))
                    return ~is_outer_bg
                profile['get_valid_mask'] = get_object_mask_white
            else:
                profile['mask_rule'] = 'Object-Based: Outer Dark Border Only'
                def get_object_mask_dark(data):
                    dark_cand = (data[0] <= 10) & (data[1] <= 10) & (data[2] <= 10)
                    labeled, _ = ndi.label(dark_cand)
                    border_labels = set(labeled[0, :]).union(set(labeled[-1, :])).union(set(labeled[:, 0])).union(set(labeled[:, -1]))
                    border_labels.discard(0)
                    is_outer_bg = np.isin(labeled, list(border_labels))
                    return ~is_outer_bg
                profile['get_valid_mask'] = get_object_mask_dark
                
        return profile

print("Model Hybrid (Pixel & Object-Based) berhasil didefinisikan!")

## 2. List dan Pilih Citra yang Ingin Diproses
Hanya satu citra yang dipilih untuk dieksekusi.

In [ ]:
# Path input dan output
input_dir = r"D:\JOB\TASK\DATA\IMAGERY\Kalbar April 2026\tif"
output_dir = r"D:\JOB\TASK\DATA\IMAGERY\Kalbar April 2026\Nodata"
os.makedirs(output_dir, exist_ok=True)

tif_files = sorted(glob.glob(os.path.join(input_dir, "*.tif*")))
print(f"Ditemukan {len(tif_files)} file di folder input:\n")
for idx, fpath in enumerate(tif_files):
    size_gb = os.path.getsize(fpath) / (1024 ** 3)
    print(f"[{idx}] {os.path.basename(fpath)} ({size_gb:.2f} GB)")

# PILIH SATU CITRA SAJA
selected_index = 0
selected_file = tif_files[selected_index]
file_name = os.path.basename(selected_file)
output_file = os.path.join(output_dir, file_name)

print(f"\n--> Citra Terpilih : {selected_file}")
print(f"--> Target Output  : {output_file}")

## 3. Jalankan Deteksi Hybrid pada Citra Terpilih

In [ ]:
profile = hybrid_detect_background(selected_file)

print("=" * 70)
print("           HASIL DETEKSI MODEL HYBRID APPROACH")
print("=" * 70)
print(f"File Analisis       : {os.path.basename(profile['file'])}")
print(f"Dimensi Citra       : {profile['width']:,} x {profile['height']:,} piksel")
print(f"Jumlah Band         : {profile['bands']} ({profile['dtype']})")
print(f"Nilai Border Tepi   : {profile['border_values']}")
print(f"Pendekatan Terpilih : {profile['approach']}")
print(f"Aturan Masking      : {profile['mask_rule']}")
if profile['has_alpha_channel']:
    print(f"Threshold Otsu      : {profile['alpha_threshold']}")
print("=" * 70)

## 4. Visualisasi Overview & Preview Masking Tegas

In [ ]:
factor = 64
with rasterio.open(selected_file) as src:
    thumb_shape = (src.count, src.height // factor, src.width // factor)
    data_thumb = src.read(out_shape=thumb_shape)

valid_mask = profile['get_valid_mask'](data_thumb)

r, g, b = data_thumb[0], data_thumb[1], data_thumb[2]
alpha_clean = np.where(valid_mask, 255, 0).astype(np.uint8)
rgba_thumb = np.dstack([r, g, b, alpha_clean])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Citra Awal
axes[0].imshow(np.dstack([r, g, b]))
axes[0].set_title("1. Citra Asli (Background Putih Terlihat)")
axes[0].axis('off')

# 2. Mask Valid Data
axes[1].imshow(valid_mask, cmap='viridis')
axes[1].set_title(f"2. Mask Valid ({profile['approach']})")
axes[1].axis('off')

# 3. Hasil Render Transparan
axes[2].set_facecolor('#1a1a1a')
axes[2].imshow(rgba_thumb)
axes[2].set_title("3. Hasil Masking (Latar Belakang Transparan)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 5. Eksekusi Export Citra Terpilih ke Folder `/Nodata` (Opsi A)
Mengekspor citra terpilih menggunakan `BIGTIFF=YES`, pemrosesan per blok/chunk `4096 x 4096`, dan kanal Alpha bersih.

In [ ]:
chunk_size = 4096

with rasterio.open(selected_file) as src:
    out_profile = src.profile.copy()
    out_profile.update({
        'driver': 'GTiff',
        'bigtiff': 'YES',         # Wajib YES untuk mendukung file berukuran > 4 GB
        'count': 4,               # 4 Band RGBA
        'photometric': 'RGB',
        'tiled': True,
        'blockxsize': 512,
        'blockysize': 512,
        'compress': 'deflate',
        'predictor': 2,
        'nodata': None
    })
    
    n_chunks_x = int(np.ceil(src.width / chunk_size))
    n_chunks_y = int(np.ceil(src.height / chunk_size))
    total_chunks = n_chunks_x * n_chunks_y
    
    print(f"Mengekspor citra terpilih : {file_name}")
    print(f"Target path              : {output_file}")
    print(f"Dimensi                  : {src.width:,} x {src.height:,} piksel ({total_chunks} chunk)\n")
    
    with rasterio.open(output_file, 'w', **out_profile) as dst:
        dst.colorinterp = [ColorInterp.red, ColorInterp.green, ColorInterp.blue, ColorInterp.alpha]
        
        with tqdm(total=total_chunks, desc=f"Menulis {file_name[:20]}") as pbar:
            for row in range(0, src.height, chunk_size):
                h = min(chunk_size, src.height - row)
                for col in range(0, src.width, chunk_size):
                    w = min(chunk_size, src.width - col)
                    win = Window(col, row, w, h)
                    
                    data = src.read(window=win)
                    
                    # Jika citra asli hanya 3 band (RGB), buat kanal Alpha baru
                    if data.shape[0] == 3:
                        alpha_init = np.full((1, h, w), 255, dtype=np.uint8)
                        data = np.concatenate([data, alpha_init], axis=0)
                    
                    is_valid = profile['get_valid_mask'](data)
                    
                    # Terapkan kanal Alpha biner murni
                    data[3] = np.where(is_valid, 255, 0).astype(np.uint8)
                    
                    # Nolkan nilai RGB di background
                    data[0, ~is_valid] = 0
                    data[1, ~is_valid] = 0
                    data[2, ~is_valid] = 0
                    
                    dst.write(data, window=win)
                    pbar.update(1)

print("\nPenulisan data citra selesai!")

## 6. Membangun Piramida Sejati (`.ovr`) untuk ArcGIS & QGIS
ArcGIS dan QGIS membutuhkan file piramida eksternal berformat **`<nama_file>.tif.ovr`** dengan urutan tingkat powers-of-2 lengkap `[2, 4, 8, 16, 32, 64, 128, 256, 512]`.

Dengan mengaktifkan konfigurasi `TIFF_USE_OVR=True` dan `COMPRESS_OVERVIEW='DEFLATE'`, GDAL/Rasterio akan membangun file `.ovr` resmi yang **langsung dikenali secara otomatis oleh ArcGIS dan QGIS tanpa memunculkan dialog/peringatan "Build Pyramids" lagi**.

In [ ]:
with rasterio.open(output_file) as src:
    # Hitung level piramida powers-of-2 lengkap sampai dimensi terkecil <= 256 piksel
    levels = []
    factor = 2
    min_dim = min(src.width, src.height)
    while min_dim // factor >= 128:
        levels.append(factor)
        factor *= 2

print(f"File target: {os.path.basename(output_file)}")
print(f"Level piramida yang dihitung secara otomatis: {levels}")
print("Membangun piramida eksternal (.ovr) terkompresi DEFLATE untuk ArcGIS/QGIS...")

# Bangun file piramida eksternal resmi ArcGIS/QGIS (.ovr)
with rasterio.Env(TIFF_USE_OVR=True, COMPRESS_OVERVIEW='DEFLATE'):
    with rasterio.open(output_file, 'r+') as dst:
        dst.build_overviews(levels, Resampling.average)

ovr_file = output_file + ".ovr"
print("\n" + "=" * 70)
print("PIRAMIDA BERHASIL DIBUAT DENGAN LENGKAP!")
print(f"File GeoTIFF : {output_file} ({os.path.getsize(output_file)/(1024**3):.2f} GB)")
if os.path.exists(ovr_file):
    print(f"File Piramida: {ovr_file} ({os.path.getsize(ovr_file)/(1024**2):.1f} MB)")
    print("--> Status: File .ovr telah tersedia. ArcGIS & QGIS tidak akan meminta build pyramid lagi!")
print("=" * 70)

## 7. Verifikasi Visual Hasil Citra di Folder `/Nodata`

In [ ]:
factor = 64
with rasterio.open(output_file) as src_out:
    shape = (src_out.count, src_out.height // factor, src_out.width // factor)
    data_res = src_out.read(out_shape=shape)
    print(f"File verifikasi : {os.path.basename(output_file)}")
    print(f"Color Interp    : {src_out.colorinterp}")
    print(f"Piramida Levels : {src_out.overviews(1)}")
    print(f"Dimensi Thumbnail: {data_res.shape}")

rgba_res = data_res.transpose(1, 2, 0)

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_facecolor('#1a1a1a')
ax.imshow(rgba_res)
ax.set_title(f"Hasil Akhir: {file_name} (Latar Transparan)", fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()